In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")


# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti 
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( puti )
importlib.reload( uti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:
from myPythonTools.YAML import get_exp_list as Xl
importlib.reload(Xl)


In [ ]:
print( sys.path )
############################
# Initialize dict for regridding
############################
RgObs={}


In [ ]:
### Make a list of files
import yaml
#from box import Box


with open('../YAML/configure_plotting.yaml', 'r') as file:
    cfgPlot = yaml.safe_load(file)


######################################################
season = 'jja' # cfgPlot['latlon']['season']
valData  = cfgPlot['latlon']['validation']['data']
valYears = cfgPlot['latlon']['validation']['years']
print( f"Season= {season}" )
print( valData , valYears )

# other ... 
#fld='TAUX' #'PRECT'
#fld='PRECT'
#fld='U'
fld='UTGWSPEC'
#fld='UTGW_MOVMTN'
#fld='RESTOM'
#fld='U10'
#fld='LHFLX'
plot_validation=True
use_desc_in_title = True #True
use_ymdPat_from_yaml = True

In [ ]:

exp_yaml,xl= Xl.fGW_tests() 


#exp_yaml='GW_Xtreme_ppe.yaml'
#xl= ['control','nofgw','NOogw','NOgwATall',] #'ne240x2_QxQsst']


with open(f'../YAML/{exp_yaml}', 'r') as file:
    cfg = yaml.safe_load(file)


In [ ]:

#exp_yaml,xl= Xl.Official()  #Utility()

#with open(f'../YAML/{exp_yaml}', 'r') as file:
#    cfg = yaml.safe_load(file)

Al =[]
for x in xl:
    exp, subd, Src, Hkey, Dst, useri = cfg[x]['name'] , cfg[x]['subdir'] , cfg[x]['SrcGrid'] , cfg[x]['Hkey'] , cfg[x]['DstGrid'] , cfg[x]['user'] 

    if (use_ymdPat_from_yaml==True):
        ymdPat=cfg[x]['ymdPat']
        if ( 'ymdPat_is_range' in cfg[x] ):
            ymdPat_is_range = cfg[x]['ymdPat_is_range']
        else:
            ymdPat_is_range = False
    else:
        ymdPat = '*' # ['1985-*', '1986-*']
        ymdPat_is_range = False

    print( exp, subd, Src, Hkey, Dst, useri )
    print( ymdPat  )
    print( ymdPat_is_range )
    
    A = uti.MakeDict4Exp( exp=exp , user=useri, subd=subd , 
                     hsPat='cam.h1a' , ymdPat=ymdPat,
                     ymdPat_is_range =ymdPat_is_range , 
                     verbose=True, open_dataset=True )
    
    A['Src']=Src
    A['Hkey']=Hkey
    if 'desc' in cfg[x]:
        A['desc'] = cfg[x]['desc']
    if 'gw_parameters' in cfg[x]:
        A['gw_parameters'] = cfg[x]['gw_parameters']
    Al.append( A )
    print( f'{A.exp} Years {len(A.X.time) /12. } \n')


In [ ]:
print(isinstance(ymdPat, list ) )
print(ymdPat)

In [ ]:
# add PRECT if not there
for A in Al:
    if ( 'PRECT' not in A.X ) and ( 'PRECL' in A.X ) and ( 'PRECC' in A.X ):
        A.X['PRECT'] = A.X.PRECC + A.X.PRECL
for A in Al:
    if ( 'RESTOM' not in A.X ) and ( 'FSNT' in A.X ) and ( 'FLNT' in A.X ):
        A.X['RESTOM'] = A.X.FSNT - A.X.FLNT


In [ ]:
Al[0].X

In [ ]:
########################################
# Make seasonal means

for A in Al:
    print( f'{A.exp} Years {len(A.X.time) /12. }')
########################################
    
    AAa_s,years_a,months_a = Av.Seasonal( ds=A.X, season=season , fld=fld,return_time=True)
    A['AA_s']=AAa_s
    A['years']=years_a
    A['months']=months_a


In [ ]:
########################################
# Regrid to lat-lon if needed
########################################
method='C2'

for A in Al:
    Hkey = A['Hkey']
    Src  = A['Src']
    AAa_s = A['AA_s']
    
    if (Hkey == 'yx' ):
        ########################
        # Option 1 from lat-lon regirrdded data
        ########################

        lon_a=A.X.lon.values
        lat_a=A.X.lat.values
        lev_a=A.X.lev.values
        zlev_a=-7.0*np.log( lev_a/1_000. )
        A['AA_s_yx'] = AAa_s

    elif (Hkey=='c'):
        ########################
        # Option 2 from unstructured SE data
        ########################
        
        Dst = 'fv1x1'
        DstInfo = GrU.gridInfo(Dst) #,Vgrid=DstVgrid)
        lat_a,lon_a = GrU.latlon( scrip= DstInfo['scrip'], Hkey=DstInfo['Hkey'] )
        lev_a=A.X.lev.values
        zlev_a=-7.0*np.log( lev_a/1_000. )
    
        RegridObj = GrU.regrid_object_lib(RgOb=RgObs, src=Src, dst=Dst)
        AAa_s_yx=RgF.Horz(xfld_Src=AAa_s , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
        A['AA_s_yx'] = AAa_s_yx

    A['lonx'] = lon_a
    A['latx'] = lat_a
    A['zlev'] = zlev_a
    A['plev'] = lev_a
    A['DateRange'] = f"{(A.X.time[0].values.item() ).strftime('%Y-%m-%d')}_{(A.X.time[-1].values.item() ).strftime('%Y-%m-%d')}"

In [ ]:

####
# Validation
####
mgrid = False

valYears='*'
Aval = Val.data(fld=fld,season=season,mgrid=mgrid,zlev=True , Years=valYears ) #, Years='1996' ) #, Years='2000' )
if (Aval['data_exists']==True):
    AAv_s_yx , zlev_v,lat_v,lon_v, years_v, data_source = Aval['aa'], Aval['lev'], Aval['lat'], Aval['lon'], Aval['years'], Aval['data_source']
else:
    plot_validation=False


In [ ]:
#print(Aval['lev'] , Aval['data_source'])


In [ ]:

lev_sel = uti.find_nearest_plev_indices(plev=Al[0].zlev , target_levels=[ 1.0 ] )


In [ ]:
shape_AA = np.shape( Al[0].AA_s_yx )
print(len(shape_AA))


In [ ]:
print(fld)

In [ ]:
print(Al[1].exp)

In [ ]:
##############
clev_prect=0.25*np.linspace(0,20,num=21)
#clev=np.linspace(0,1.0,num=21)

plot_Z_km=60. #0.10 #0.25 #0.9 #5. #12. #1.0

clev_fld , dlev_fld, scale_field, the_cmap = puti.contour_intervals( fld, return_cmap=True, zlev=plot_Z_km, yx=True )  #np.linspace(-0.3, 0.3 ,num=21) # 21

fields=[]
lons=[]
lats=[]
clevs=[]
titles=[]
cmaps=[]
insets=[]
control_diff = False #True
tri_panel_diff = False #True
ipanel=0
z_ref = r'$z_{ref}$'
for A in Al:
    years_a = A['years']
    if ( ( control_diff==True ) and ( ipanel>0) ):
        diff_fac = 1
        the_cmap = 'bwr'
        clev = 0.5*np.linspace(-100,100,num=21)
    else :
        diff_fac = 0
        #the_cmap = 'gist_ncar' #'Blues_r' #'gist_ncar'
        clev = clev_fld
    
    if ( len( np.shape( A.AA_s_yx ) ) == 3 ):
        kplot = uti.find_nearest_plev_indices(plev=A.zlev , target_levels=[ plot_Z_km ] )
        #fields.append(  scale_field * A.AA_s_yx[kplot[0],:,:] )
        field_in_A =  scale_field * A.AA_s_yx[kplot[0],:,:] 
        fldoo=f'{fld} ({z_ref}={A.zlev[kplot[0]]:.3f} km)'
        #fldoo=f'{fld} (z={plot_Z_km:.2f}km, level={A.zlev[kplot[0]]:.3f})'
    elif ( len( np.shape( A.AA_s_yx ) ) == 2 ):
        #fields.append(  scale_field * ( A.AA_s_yx[:,:] - diff_fac*Al[0].AA_s_yx[:,:]) )
        field_in_A = scale_field * ( A.AA_s_yx[:,:] - diff_fac*Al[0].AA_s_yx[:,:]) 
        fldoo=f'{fld} '

    ny,nx = np.shape(  field_in_A  ) 
    fields.append(  field_in_A  )
    # make weights
    lat_wt = np.cos(np.deg2rad(A.latx))   # shape (ny,)
    # expand to (ny, nx) for broadcasting
    lat_wt_yx = np.tile( lat_wt[:, np.newaxis], (1, nx))  # shape (ny, nx)
    global_mean_field = np.sum( field_in_A * lat_wt_yx ) / np.sum( lat_wt_yx )
    print( f'Global mean {A.exp} {fld} = {global_mean_field}' )
       
    lons.append( A.lonx )
    lats.append( A.latx )
    clevs.append( clev )
    if ( ( control_diff==True ) and ( ipanel>0) ):
        if ('desc' in A) and (use_desc_in_title == True):
            titles.append( f"<{A.desc}> - <{Al[0].desc}>: \n {fldoo} \n {season.upper()}  {years_a[0]}-{years_a[-1]}" )
        else:
            titles.append( f"{A.exp} - {Al[0].exp}: \n {fldoo} \n {season.upper()}  {years_a[0]}-{years_a[-1]}" )
    else:
        if ('desc' in A) and (use_desc_in_title == True):
            titles.append( f"{A.desc}: \n {fldoo} \n {season.upper()}  {years_a[0]}-{years_a[-1]}" )
        else:
            titles.append( f"{A.exp}: \n {fldoo} \n {season.upper()}  {years_a[0]}-{years_a[-1]}" )
   
    cmaps.append( the_cmap )



    
    if 'gw_parameters' in A:
        parameters=A['gw_parameters']
        # Parameters to display in the inset box
        selected_keys = ['alpha_gw_movmtn', 'effgw_rdg_resid','use_gw_rdg_resid']
        selected_params = {key: parameters[key] for key in selected_keys if key in parameters}
        # Format selected parameters into a string
        param_text = "\n".join([f"{key}: {value}" for key, value in selected_params.items()])
        inset={'text':param_text, 'x':0.65,'y':0.98}
        insets.append( inset )

    ipanel = ipanel+1

if (tri_panel_diff==True):
    if ( len( np.shape( Al[0].AA_s_yx ) ) == 3 ):
        kplot = uti.find_nearest_plev_indices(plev=A.zlev , target_levels=[ plot_Z_km ] )
        #fields.append(  scale_field * A.AA_s_yx[kplot[0],:,:] )
        field_in_A =  scale_field * ( Al[1].AA_s_yx[kplot[0],:,:] - Al[0].AA_s_yx[kplot[0],:,:] )
        fldoo=f'{fld} ({z_ref}={A.zlev[kplot[0]]:.3f} km)'
        #fldoo=f'{fld} (z={plot_Z_km:.2f}km, level={A.zlev[kplot[0]]:.3f})'
    elif ( len( np.shape( Al[0].AA_s_yx ) ) == 2 ):
        #fields.append(  scale_field * ( A.AA_s_yx[:,:] - diff_fac*Al[0].AA_s_yx[:,:]) )
        field_in_A = scale_field * ( Al[1].AA_s_yx[:,:] - Al[0].AA_s_yx[:,:] ) 
        fldoo=f'{fld} '

    fields.append(  field_in_A  )
    # make weights
    lat_wt = np.cos(np.deg2rad(A.latx))   # shape (ny,)
    # expand to (ny, nx) for broadcasting
    lat_wt_yx = np.tile( lat_wt[:, np.newaxis], (1, nx))  # shape (ny, nx)
    global_mean_field = np.sum( field_in_A * lat_wt_yx ) / np.sum( lat_wt_yx )
    print( f'Global mean Diff = {global_mean_field}' )
       
    lons.append( A.lonx )
    lats.append( A.latx )
    clevs.append( dlev_fld )
    cmaps.append( 'bwr' )
    
    titles.append( f" Diff ( b - a )" )


if (plot_validation==True):
    ##
    # append validation
    if (len(zlev_v) >1):
        kplot = uti.find_nearest_plev_indices(plev=zlev_v , target_levels=[ plot_Z_km ] )
        fields.append(  AAv_s_yx[kplot[0],:,:] )
        fldoo=f'{fld} ({z_ref}={zlev_v[kplot[0]]:.3f} km)'
        #fldoo=f'{fld} (z={plot_Z_km:.2f}km, level={zlev_v[kplot[0]]:.3f})'
    else:
        fields.append(  AAv_s_yx[:,:] )
    lons.append( lon_v )
    lats.append( lat_v )
    clevs.append( clev )
    titles.append( f"{fldoo} {season.upper()} {years_v} {data_source}" )
    cmaps.append( the_cmap )


In [ ]:
print( titles )

In [ ]:
####
MapProj = ccrs.PlateCarree(central_longitude=180.)
map_extent =[10,150,0,50]
map_extent = None #[-140,-0.1,-50,15]

LL.Maps( fields=fields ,lons=lons ,lats=lats ,clevs=clevs,cmaps=cmaps , titles=titles, insets=insets, Projection=MapProj, Landmask=False ,map_extent=map_extent )

In [ ]:
lat0,lon0=-50,270
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
print(y,x)

In [ ]:
A.X

In [ ]:
for A in Al:
    AA_tR = A.X[fld].values
    #nt,nz,ny,nx = np.shape( AA_tR )
    nt,ny,nx = np.shape( AA_tR )
    #A['AA_tR'] = AA_tR.reshape( (nt//12, 12, nz,ny,nx ) )
    A['AA_tR'] = AA_tR.reshape( (nt//12, 12,ny,nx ) )
    A['AA_YR'] = np.average( A['AA_tR'] , axis=1 )
    A['AA_YRs'] = np.std( A['AA_YR'] , axis=0 )
    A['AA_YRm'] = np.mean( A['AA_YR'] , axis=0 )




In [ ]:
lat0,lon0=-50,90
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
print( lon0,lat0)
n=0
print( Al[n].AA_YRm[y,x]-Al[n].AA_YRs[y,x], Al[n].AA_YRm[y,x] ,Al[n].AA_YRm[y,x]+Al[n].AA_YRs[y,x]  )
n=1
print( Al[n].AA_YRm[y,x]-Al[n].AA_YRs[y,x], Al[n].AA_YRm[y,x] ,Al[n].AA_YRm[y,x]+Al[n].AA_YRs[y,x]  )

lat0,lon0=-50,200
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
print( lon0,lat0)
n=0
print( Al[n].AA_YRm[y,x]-Al[n].AA_YRs[y,x], Al[n].AA_YRm[y,x] ,Al[n].AA_YRm[y,x]+Al[n].AA_YRs[y,x]  )
n=1
print( Al[n].AA_YRm[y,x]-Al[n].AA_YRs[y,x], Al[n].AA_YRm[y,x] ,Al[n].AA_YRm[y,x]+Al[n].AA_YRs[y,x]  )

lat0,lon0=-50,270
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
print( lon0,lat0)
n=0
print( Al[n].AA_YRm[y,x]-Al[n].AA_YRs[y,x], Al[n].AA_YRm[y,x] ,Al[n].AA_YRm[y,x]+Al[n].AA_YRs[y,x]  )
n=1
print( Al[n].AA_YRm[y,x]-Al[n].AA_YRs[y,x], Al[n].AA_YRm[y,x] ,Al[n].AA_YRm[y,x]+Al[n].AA_YRs[y,x]  )




In [ ]:
lat0,lon0=-50,200
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
plt.plot( Al[0]['AA_YRs'][:,y,x] )
plt.plot( Al[1]['AA_YRs'][:,y,x] )


In [ ]:
print( np.shape( Al[1].AA_tR ) )

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(18,5))

ax=axs[0]
lat0,lon0=-50,90
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
ax.plot( Al[0].AA_s_yx[:,y,x], zlev_a, label=Al[0].desc )
mean,std = Al[0]['AA_YRm'][:,y,x], Al[0]['AA_YRs'][:,y,x]
ax.fill_betweenx(zlev_a, mean[:] - 1*std[:], mean[:] + 1*std[:] ,
                 alpha=0.3, label="±1 Std Dev")

ax.plot( Al[1].AA_s_yx[:,y,x], zlev_a,  label=Al[1].desc )
mean,std = Al[1]['AA_YRm'][:,y,x], Al[1]['AA_YRs'][:,y,x]
ax.fill_betweenx(zlev_a, mean[:] - 1*std[:], mean[:] + 1*std[:] ,
                 alpha=0.4, label="±1 Std Dev")

ax.plot( AAv_s_yx[:,y,x], zlev_v , label='ERA5' )
ax.legend()
ax.set_ylim(0,2)
ax.set_xlim(8,20)
ax.set_xlabel(f'{fld}' )
ax.set_ylabel(f'Z (km)' )
ax.set_title(f'lon={lon0} lat={lat0}' )

ax=axs[1]
lat0,lon0=-50,200
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
ax.plot( Al[0].AA_s_yx[:,y,x], zlev_a, label=Al[0].desc )
mean,std = Al[0]['AA_YRm'][:,y,x], Al[0]['AA_YRs'][:,y,x]
ax.fill_betweenx(zlev_a, mean[:] - 1*std[:], mean[:] + 1*std[:] ,
                 alpha=0.3, label="±1 Std Dev")

ax.plot( Al[1].AA_s_yx[:,y,x], zlev_a,  label=Al[1].desc )
mean,std = Al[1]['AA_YRm'][:,y,x], Al[1]['AA_YRs'][:,y,x]
ax.fill_betweenx(zlev_a, mean[:] - 1*std[:], mean[:] + 1*std[:] ,
                 alpha=0.4, label="±1 Std Dev")

ax.plot( AAv_s_yx[:,y,x], zlev_v , label='ERA5' )

ax.legend()
ax.set_ylim(0,2)
ax.set_xlim(5,15)
ax.set_xlabel(f'{fld}' )
ax.set_ylabel(f'Z (km)' )
ax.set_title(f'lon={lon0} lat={lat0}' )

ax=axs[2]
lat0,lon0=-50,270
y = np.argmin( np.abs( Al[0].latx - lat0 ) )
x = np.argmin( np.abs( Al[0].lonx - lon0 ) )
ax.plot( Al[0].AA_s_yx[:,y,x], zlev_a, label=Al[0].desc )
mean,std = Al[0]['AA_YRm'][:,y,x], Al[0]['AA_YRs'][:,y,x]
ax.fill_betweenx(zlev_a, mean[:] - 1*std[:], mean[:] + 1*std[:] ,
                 alpha=0.3, label="±1 Std Dev")
ax.plot( Al[1].AA_s_yx[:,y,x], zlev_a,  label=Al[1].desc )
mean,std = Al[1]['AA_YRm'][:,y,x], Al[1]['AA_YRs'][:,y,x]
ax.fill_betweenx(zlev_a, mean[:] - 1*std[:], mean[:] + 1*std[:] ,
                 alpha=0.4, label="±1 Std Dev")
ax.plot( AAv_s_yx[:,y,x], zlev_v , label='ERA5' )
ax.legend()
ax.set_ylim(0,2)
ax.set_xlim(5,15)
ax.set_xlabel(f'{fld}' )
ax.set_ylabel(f'Z (km)' )
ax.set_title(f'lon={lon0} lat={lat0}' )



In [ ]:
print(np.shape(AAv_s_yx))
print(np.shape(Al[0].X.U.values))



In [ ]:
print( np.shape( fields[0] ) )
plt.contour( lat_wt_yx )

In [ ]:
plt.plot(A.X.w)

In [ ]:
Al[1].AA_s_yx.shape

In [ ]:

topoF=Al[1].X.topography_file
topo=xr.open_dataset( topoF )
sgh=topo.SGH.values
latc=topo.lat.values
lonc=topo.lon.values


In [ ]:
plt.tricontourf( lonc, latc, sgh , levels=[1,100,200,500,1000],cmap='terrain' )
plt.colorbar()

In [ ]:
####
kplot = uti.find_nearest_plev_indices(plev=Al[0].zlev , target_levels=[ plot_Z_km ] )
print(Al[0].zlev[kplot])
print(Al[0].plev[kplot])


In [ ]:

am=Al[1].X.hyam.values
bm=Al[1].X.hybm.values
ai=Al[1].X.hyai.values
bi=Al[1].X.hybi.values
ps=Al[1].X.PS.values




pmid,pint,delp = MkP.Pressure ( am=am, bm=bm , ai=ai, bi=bi, ps=ps , p_00=100_000., Gridkey='tzc' )

cldliq=Al[1].X.CLDLIQ.values
nt,nz,ncol=np.shape( cldliq )
lwp2=np.zeros( (nt, ncol ) )

print( np.shape( lwp2 ) )

for m in np.arange( 12 ):
    lwp2[m,:] = np.sum( delp[m,:,:] * cldliq[m,:,:] , axis=0 )


In [ ]:
lwp1=Al[1].X.TGCLDLWP.values


In [ ]:
print( np.shape(lwp1) , np.shape(lwp2) )

plt.scatter( 9.8*lwp1[0,:],lwp2[0,:] )





In [ ]:
plt.tricontourf( lonc,latc, 9.8*lwp1[0,:]-lwp2[0,:] )

In [ ]:
#plt.plot( np.average( scale_field*Al[0].AA_s_yx , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*Al[0].AA_s_yx , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*Al[1].AA_s_yx, axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*Al[2].AA_s_yx, axis=1 ) )


In [ ]:
#plt.plot( np.average( scale_field*Al[0].AA_s_yx , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*(Al[1].AA_s_yx -Al[0].AA_s_yx) , axis=1 ) )
plt.plot( Al[0].latx, np.average( scale_field*(Al[2].AA_s_yx -Al[0].AA_s_yx), axis=1 ) )


In [ ]:
#####